# SARBLOH main: Duck harness on a fast, fault-tolerant vLLM

The submission notebook. **The agent is Tufa Labs' Duck, unchanged**: it runs from their public source bundle
(`jeroencottaar/taaf-kaggle-source-share`) with its pickled benchmark, solver and scorer. Full credit for the solver
and the harness goes to Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit, Michal Tesnar and Stefano Viel
(Tufa Labs).

What this notebook changes is **serving only**:

1. **Our own vLLM launcher** replaces the bundle's `setup_commands.json`. It exports exactly the same
   `LOCAL_ANALYZER_*` environment, so the Duck sees no difference.
2. **Profile chain.** vLLM profiles are tried in order (`fast`, then `fast_nospec`, then `duck`). The last one is
   the Duck's exact flags. A profile is accepted only if the server starts **and** passes a tool-call smoke test,
   so a failed speed experiment costs minutes, not the run.
3. **Throughput probe.** After start-up, a concurrent generation burst measures aggregate tok/s for the ledger.
4. **Watchdog.** It polls `/v1/models`. After 4 consecutive failures it restarts the active profile, at most twice.
5. **Guaranteed teardown.** On success or on any error, the server is stopped, the 10+ GB vLLM install is
   removed from `/kaggle/working`, and an offline run always leaves a `submission.parquet`.

Everything is set from the `CONFIG` cell. Attach: the competition, the three datasets named in `CONFIG`, and the
RTX Pro 6000 accelerator. Internet stays **off**.

## 0. CONFIG: every knob is here

In [ ]:
CONFIG = {
    "experiment": "E001_duck_fast_vllm",
    # --- Kaggle inputs (attach all three) --------------------------------------------------------------------
    "bundle_marker": "taaf-kaggle-bundle.json",                        # identifies the Duck source bundle
    "bundle_dataset": "jeroencottaar/taaf-kaggle-source-share",
    "wheelhouse_dataset": "driessmit1/arc3-vllm-h100-wheelhouse-v3",   # or our own: see kaggle/wheels.ipynb
    "wheelhouse_stamp": "vllm==0.19.0 torch==2.10.0 flashinfer==0.6.6\n",
    "model_dataset": "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot",
    "served_model_name": "vrfai/Qwen3.6-27B-FP8",
    "port": 1234,
    # --- vLLM ------------------------------------------------------------------------------------------------
    "vllm_profile_chain": ["fast_nospec", "duck"],   # tried in order; keep "duck" last. "fast" (MTP) dropped: 0% draft acceptance, this snapshot has no MTP weights (2026-09-25)
    "vllm_startup_timeout_s": 1500,
    "vllm_bench": {"enabled": True, "concurrency": 16, "max_tokens": 512},
    "watchdog": {"enabled": True, "interval_s": 15.0, "failures_to_restart": 4, "max_restarts": 2},
    # --- Duck client: identical to the bundle's setup command ------------------------------------------------
    "analyzer_context_window": 32768,
    # --- Solver overrides applied to the unpickled benchmark (None = keep the pickled value) ----------------
    "solver": {
        "max_runtime_s_per_game": 2400.0,   # QUICK CHECK; full run: 7920.0
        "analyzer_timeout": 900.0,
        "concurrency": 28,
        "max_actions_per_game": None,
        "save_request_logs": False,
    },
    # --- Budget ------------------------------------------------------------------------------------------------
    "budget_s": 5400.0,            # QUICK CHECK (1.5 h); full E001 run: 32400.0 (the Duck's 9 h); hard Kaggle limit 12 h
    "teardown_reserve_s": 600.0,   # the solver is cancelled this long before budget_s
    # --- Offline (Save & Run) only: None = all competition env files; else a list of game ids or prefixes ----
    "offline_games": ["ls20", "vc33"],   # QUICK CHECK; set None for the full E001 run
}

# vLLM server profiles. "duck" reproduces the bundle's setup command exactly; the others add speed flags.
# UNCONFIRMED until E001 measures them: whether vLLM 0.19 + this Qwen3.6-27B-FP8 snapshot supports MTP,
# and whether each flag helps. A profile that fails to start or fails the smoke test falls through to the next.
DUCK_FLAGS = {
    "--tensor-parallel-size": "1",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "qwen3_coder",
    "--generation-config": "vllm",
    "--enable-prefix-caching": None,
    "--default-chat-template-kwargs": '{"preserve_thinking": true}',
    "--reasoning-parser": "qwen3",
    "--max-model-len": "65536",
}
VLLM_PROFILES = {
    "duck": {"flags": dict(DUCK_FLAGS), "env": {}},
    # Batching knobs only: 28 Duck games decode concurrently, so larger scheduled batches, with CUDA graphs covering them.
    "fast_nospec": {
        "flags": {**DUCK_FLAGS, "--max-num-seqs": "32", "--max-num-batched-tokens": "8192"},
        "env": {"OMP_NUM_THREADS": "1"},
    },
    # DuckQwen's measured winner, transplanted to this model: 3-token native MTP, prefix caching off.
    "fast": {
        "flags": {
            **{k: v for k, v in DUCK_FLAGS.items() if k != "--enable-prefix-caching"},
            "--no-enable-prefix-caching": None,
            "--speculative-config": '{"method": "mtp", "num_speculative_tokens": 3}',
            "--max-num-seqs": "32",
            "--max-num-batched-tokens": "8192",
        },
        "env": {"OMP_NUM_THREADS": "1"},
    },
}

## 1. Environment, submission mode, and the facts this run settles

In [ ]:
import json
import os
import pickle
import shutil
import signal
import subprocess
import sys
import sysconfig
import threading
import time
import traceback
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
COMPETITION_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"          # RESET keeps the level (pinned before arc_agi builds its client)
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if e
)


def elapsed() -> str:
    return f"t+{time.time() - NOTEBOOK_START_EPOCH:.0f}s"


def log(msg: str) -> None:
    print(f"[main {elapsed()}] {msg}", flush=True)


log(f"TRUE_SUBMISSION={TRUE_SUBMISSION} experiment={CONFIG['experiment']}")
if not TRUE_SUBMISSION:
    for cmd in (["nvidia-smi"], ["df", "-h", "/kaggle/working"], ["free", "-g"], ["nproc"]):
        try:
            print(subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout)
        except Exception as exc:  # noqa: BLE001
            print(f"{cmd[0]} unavailable: {exc!r}")
    print("python", sys.version)
    print("inputs:", sorted(str(p) for p in Path("/kaggle/input").glob("*/*")))

## 2. Install the ARC runtime (competition wheels, offline)

In [ ]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
     "--disable-pip-version-check", "--find-links", str(COMPETITION_DIR / "arc_agi_3_wheels"), "arc-agi"],
    stdout=subprocess.DEVNULL,
)
import arcengine  # noqa: E402

log(f"GameAction = {[a.name for a in arcengine.GameAction]}")

## 3. Locate inputs and make the Duck bundle importable (the bundle's setup commands are *not* run)

In [ ]:
def find_input(ref: str) -> Path:
    """Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>."""
    owner, slug = ref.split("/", 1)
    for p in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug):
        if p.exists():
            return p
    hits = [p for p in Path("/kaggle/input").glob(f"**/{slug}") if p.is_dir()]
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Kaggle input {ref!r} is not attached (Add Input -> Datasets).")


def find_model_dir(root: Path) -> Path:
    if (root / "config.json").exists():
        return root
    hits = sorted(root.glob("**/config.json"), key=lambda p: len(p.parts))
    if not hits:
        raise FileNotFoundError(f"No config.json under {root}")
    return hits[0].parent


markers = list(Path("/kaggle/input").rglob(CONFIG["bundle_marker"]))
if not markers:
    raise RuntimeError(f"Duck source bundle not found: attach {CONFIG['bundle_dataset']}")
BUNDLE_DIR = markers[0].parent
WHEELHOUSE = find_input(CONFIG["wheelhouse_dataset"])
MODEL_DIR = find_model_dir(find_input(CONFIG["model_dataset"]))
if not (WHEELHOUSE / "requirements.lock").exists():
    raise FileNotFoundError(f"{WHEELHOUSE}/requirements.lock missing: wrong wheelhouse dataset?")
log(f"bundle={BUNDLE_DIR} wheelhouse={WHEELHOUSE} model={MODEL_DIR}")

# The same env contract the bundle's setup/solver expect.
kaggle_input_paths = {
    CONFIG["bundle_dataset"]: str(BUNDLE_DIR),
    CONFIG["wheelhouse_dataset"]: str(WHEELHOUSE),
    CONFIG["model_dataset"]: str(find_input(CONFIG["model_dataset"])),
}
os.environ.update({
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(list(kaggle_input_paths)),
    "TAAF_KAGGLE_KERNEL_SOURCES": "[]",
    "TAAF_KAGGLE_BUNDLE_DIR": str(BUNDLE_DIR),
    "TAAF_KAGGLE_WORKING_DIR": str(WORKING_DIR),
})

source_entries = []
for repo in sorted((BUNDLE_DIR / "src").iterdir(), reverse=True):
    for candidate in (repo / "src", repo):
        if candidate.is_dir():
            source_entries.append(candidate)
for entry in source_entries:
    sys.path.insert(0, str(entry))
(Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth").write_text("".join(f"{e}\n" for e in source_entries))
log(f"{len(source_entries)} source roots on sys.path")

## 4. vLLM: install, then walk the profile chain until one starts **and** passes the smoke test

This is the bundle's setup command rewritten with profiles, early crash detection, a tool-call smoke test and a
throughput probe. The analyzer environment it exports is byte-identical to the bundle's.

In [ ]:
SITE_PACKAGES = WORKING_DIR / "vllm-site-packages"
VLLM_LOG = WORKING_DIR / "vllm-server.log"
VLLM_BASE_URL = f"http://127.0.0.1:{CONFIG['port']}/v1"
SERVE = {"process": None, "profile": None, "restarts": 0, "bench": {}, "attempts": []}


def vllm_env(profile: str) -> dict:
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join(p for p in (str(SITE_PACKAGES), env.get("PYTHONPATH", "")) if p)
    env.update({"USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1"})
    env.update(VLLM_PROFILES[profile]["env"])
    return env


def install_vllm() -> None:
    stamp = SITE_PACKAGES / ".sarbloh-stamp"
    if stamp.exists() and stamp.read_text(encoding="utf-8") == CONFIG["wheelhouse_stamp"]:
        log("vLLM install cached")
        return
    shutil.rmtree(SITE_PACKAGES, ignore_errors=True)
    SITE_PACKAGES.mkdir(parents=True)
    t = time.time()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(WHEELHOUSE),
         "--requirement", str(WHEELHOUSE / "requirements.lock"), "--target", str(SITE_PACKAGES),
         "--upgrade", "--ignore-installed", "--only-binary", ":all:", "--no-compile",
         "--disable-pip-version-check", "--no-warn-conflicts", "--quiet"],
        check=True,
    )
    check = subprocess.run([sys.executable, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"],
                           env=vllm_env("duck"), capture_output=True, text=True)
    if check.returncode != 0:
        raise RuntimeError(f"vLLM import failed after install:\n{check.stderr[-3000:]}")
    stamp.write_text(CONFIG["wheelhouse_stamp"], encoding="utf-8")
    log(f"vLLM installed in {time.time() - t:.0f}s: vllm/torch = {check.stdout.strip()}")


def request_json(url: str, payload: dict | None = None, timeout: float = 30) -> dict:
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def healthy(timeout: float = 5) -> bool:
    try:
        request_json(f"{VLLM_BASE_URL}/models", timeout=timeout)
        return True
    except Exception:  # noqa: BLE001
        return False


def log_tail(lines: int = 60) -> str:
    if not VLLM_LOG.exists():
        return ""
    return "\n".join(VLLM_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


def stop_server() -> None:
    proc = SERVE["process"]
    SERVE["process"] = None
    if proc is None or proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)       # vLLM spawns engine workers; kill the whole group
    except Exception:  # noqa: BLE001
        proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except Exception:  # noqa: BLE001
            proc.kill()
    # Wait for the port to free so a restart can bind it.
    for _ in range(30):
        if not healthy(timeout=2):
            break
        time.sleep(2)


def launch(profile: str) -> None:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(MODEL_DIR),
           "--served-model-name", CONFIG["served_model_name"], "--host", "127.0.0.1", "--port", str(CONFIG["port"])]
    for flag, value in VLLM_PROFILES[profile]["flags"].items():
        cmd += [flag] if value is None else [flag, value]
    with VLLM_LOG.open("a", encoding="utf-8") as fh:
        fh.write(f"\n===== profile={profile} {time.ctime()} =====\n{' '.join(cmd)}\n")
    log(f"starting vLLM profile={profile}")
    SERVE["process"] = subprocess.Popen(cmd, env=vllm_env(profile), stdout=VLLM_LOG.open("a", encoding="utf-8"),
                                        stderr=subprocess.STDOUT, text=True, start_new_session=True)
    SERVE["profile"] = profile


def wait_ready(timeout_s: float) -> bool:
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        proc = SERVE["process"]
        if proc is None or proc.poll() is not None:
            return False                           # crashed: fail fast instead of waiting out the timeout
        if healthy():
            return True
        time.sleep(5)
    return False


def smoke_test() -> None:
    """The Duck depends on the tool-call and reasoning parsers, so check both, not just text generation."""
    tools = [{"type": "function", "function": {
        "name": "python", "description": "Run python code.",
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}}]
    payload = {"model": CONFIG["served_model_name"], "temperature": 0.0, "max_tokens": 512, "tools": tools,
               "chat_template_kwargs": {"enable_thinking": False},   # thinking could exhaust max_tokens first
               "messages": [{"role": "user", "content": "Use the python tool to compute 17*23. Call the tool."}]}
    t = time.time()
    out = request_json(f"{VLLM_BASE_URL}/chat/completions", payload, timeout=600)
    msg = out["choices"][0]["message"]
    calls = msg.get("tool_calls") or []
    if not calls or calls[0]["function"]["name"] != "python":
        raise RuntimeError(f"smoke test: no python tool call parsed: {json.dumps(msg)[:1500]}")
    json.loads(calls[0]["function"]["arguments"])
    log(f"smoke ok in {time.time() - t:.1f}s: tool_call={calls[0]['function']['arguments'][:120]!r} "
        f"reasoning_chars={len(msg.get('reasoning_content') or msg.get('reasoning') or '')}")


def throughput_probe() -> dict:
    cfg = CONFIG["vllm_bench"]
    payload = {"model": CONFIG["served_model_name"], "temperature": 0.6, "top_p": 0.95, "max_tokens": cfg["max_tokens"],
               "ignore_eos": True, "chat_template_kwargs": {"enable_thinking": False},
               "messages": [{"role": "user", "content": "Describe a 64x64 grid puzzle game in detail."}]}

    def one(_):
        out = request_json(f"{VLLM_BASE_URL}/chat/completions", payload, timeout=900)
        return (out.get("usage") or {}).get("completion_tokens", 0)

    single_t = time.time()
    single = one(0)
    single_tps = single / max(1e-6, time.time() - single_t)
    t = time.time()
    with ThreadPoolExecutor(cfg["concurrency"]) as pool:
        total = sum(pool.map(one, range(cfg["concurrency"])))
    result = {"single_stream_tok_s": round(single_tps, 1),
              "aggregate_tok_s": round(total / max(1e-6, time.time() - t), 1), "concurrency": cfg["concurrency"]}
    log(f"throughput profile={SERVE['profile']}: {result}")
    return result


def start_vllm() -> None:
    install_vllm()
    for profile in CONFIG["vllm_profile_chain"]:
        t = time.time()
        launch(profile)
        last = profile == CONFIG["vllm_profile_chain"][-1]
        ok, err = wait_ready(CONFIG["vllm_startup_timeout_s"]), ""
        if ok:
            try:
                smoke_test()
            except Exception as exc:  # noqa: BLE001
                err = repr(exc)
                # The last profile is the reference (the Duck's flags): a server that is up is accepted even if the
                # stricter smoke test fails, exactly as the original launcher would have run it.
                ok = last
                log(f"smoke test failed on profile={profile}: {err[:800]}" + (" (accepted: last profile)" if last else ""))
        else:
            err = "did not become ready"
        SERVE["attempts"].append({"profile": profile, "ok": ok, "seconds": round(time.time() - t), "error": err[:500]})
        if ok:
            log(f"vLLM READY profile={profile} after {time.time() - t:.0f}s")
            return
        log(f"profile={profile} FAILED ({err}); log tail:\n{log_tail()}")
        stop_server()
    raise RuntimeError(f"vLLM failed on every profile: {SERVE['attempts']}")


start_vllm()
if CONFIG["vllm_bench"]["enabled"]:
    try:
        SERVE["bench"] = throughput_probe()
    except Exception as exc:  # noqa: BLE001  (a probe failure must never cost the run)
        log(f"throughput probe failed: {exc!r}")

# Exactly the analyzer environment the bundle's setup command exports.
ANALYZER_ENV = {
    "USE_TF": "0", "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_TORCHVISION": "1", "VLLM_NO_USAGE_STATS": "1",
    "PYTHONPATH": str(SITE_PACKAGES) + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "LOCAL_ANALYZER_BASE_URL": VLLM_BASE_URL, "OPENAI_BASE_URL": VLLM_BASE_URL,
    "LOCAL_ANALYZER_PROVIDER": "vllm", "OPENAI_PROVIDER": "vllm",
    "LOCAL_ANALYZER_MODEL_ID": CONFIG["served_model_name"], "INFERENCE_ANALYZER_MODEL": CONFIG["served_model_name"],
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": str(CONFIG["analyzer_context_window"]),
    "LOCAL_ANALYZER_MAX_OUTPUT": "0", "LOCAL_ANALYZER_TOOL_STEPS": "0", "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024", "LOCAL_ANALYZER_YIELD_SECONDS": "60",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6", "LOCAL_ANALYZER_TOP_P": "0.95", "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_ENABLE_THINKING": "true", "MULTIMODAL_CONTEXT": "current_grid", "MULTIMODAL_UPSCALE": "4",
}
os.environ.update(ANALYZER_ENV)
for entry in reversed([e for e in os.environ["PYTHONPATH"].split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Watchdog: restart the active profile after sustained health-check failures

In [ ]:
WATCHDOG_STOP = threading.Event()


def watchdog_loop(cfg: dict) -> None:
    failures = 0
    while not WATCHDOG_STOP.wait(cfg["interval_s"]):
        proc = SERVE["process"]
        dead = proc is None or proc.poll() is not None
        if not dead and healthy():
            failures = 0
            continue
        failures += 1
        if (dead or failures >= cfg["failures_to_restart"]) and SERVE["restarts"] < cfg["max_restarts"]:
            SERVE["restarts"] += 1
            log(f"[watchdog] restart #{SERVE['restarts']} profile={SERVE['profile']} dead={dead} failures={failures}\n{log_tail(30)}")
            try:
                stop_server()
                launch(SERVE["profile"])
                log(f"[watchdog] ready={wait_ready(CONFIG['vllm_startup_timeout_s'])}")
            except Exception as exc:  # noqa: BLE001
                log(f"[watchdog] restart failed: {exc!r}")
            failures = 0


if CONFIG["watchdog"]["enabled"]:
    threading.Thread(target=watchdog_loop, args=(CONFIG["watchdog"],), name="vllm-watchdog", daemon=True).start()
    log("watchdog started")

## 6. Load the Duck's pickled benchmark and apply the CONFIG overrides

In [ ]:
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as fh:
    target = pickle.load(fh)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as fh:
    bm = pickle.load(fh)
bm.job_dir = WORKING_DIR
for key, value in CONFIG["solver"].items():
    if value is not None or key == "max_actions_per_game":
        setattr(bm.solver, key, value)
log("solver: " + ", ".join(f"{k}={getattr(bm.solver, k, '?')}" for k in CONFIG["solver"]))

## 7. Play: live gateway on a competition rerun, bundled env files otherwise; teardown always runs

In [ ]:
import arc_agi  # noqa: E402
import taaf.game_api  # noqa: E402


def make_games(mode, base_url: str = "", env_dir: str = ""):
    spec = taaf.game_api.ArcadeSpec(operation_mode=mode, arc_base_url=base_url, environments_dir=env_dir) \
        if base_url else taaf.game_api.ArcadeSpec(operation_mode=mode, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=mode, arc_base_url=base_url, environments_dir=env_dir) \
        if base_url else arc_agi.Arcade(operation_mode=mode, environments_dir=env_dir)
    ids = [e.game_id for e in arcade.available_environments]
    if not ids:
        raise RuntimeError(f"Arcade exposed zero environments (mode={mode}).")
    return [taaf.game_api.GameAPI(env_name=i, arcade_spec=spec) for i in ids]


def wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline, last = time.monotonic() + timeout_s, ""
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(f"{base_url}api/games", timeout=10) as resp:
                if resp.status < 500:
                    return
        except Exception as exc:  # noqa: BLE001
            last = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last}")


def write_placeholder_submission() -> None:
    import pandas as pd

    pd.DataFrame([["1_0", "1", True, 1]], columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
        WORKING_DIR / "submission.parquet", index=False)


def teardown() -> None:
    WATCHDOG_STOP.set()
    stop_server()
    shutil.rmtree(SITE_PACKAGES, ignore_errors=True)   # keep /kaggle/working small: this install is 10+ GB
    log("teardown done")


os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
RUN_ERROR = None
try:
    if TRUE_SUBMISSION:
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        wait_for_gateway(os.environ["ARC_BASE_URL"])
        bm.games = make_games(arc_agi.OperationMode.COMPETITION, base_url=os.environ["ARC_BASE_URL"])
    else:
        games = make_games(arc_agi.OperationMode.OFFLINE, env_dir=str(COMPETITION_DIR / "environment_files"))
        only = CONFIG["offline_games"]
        if only:
            games = [g for g in games if any(g.env_name == o or g.env_name.startswith(o) for o in only)]
        bm.games = games
    bm.n_passes = 1
    bm.game_weights = None
    log(f"playing {len(bm.games)} games")

    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
        seconds=CONFIG["budget_s"] - CONFIG["teardown_reserve_s"])
    log(f"soft end at {soft_end:%H:%M:%S} ({CONFIG['budget_s'] - CONFIG['teardown_reserve_s']:.0f}s after start)")
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
except Exception as exc:  # noqa: BLE001
    RUN_ERROR = exc
    traceback.print_exc()
finally:
    try:
        if not TRUE_SUBMISSION:
            write_placeholder_submission()   # Save & Run is not scored but must leave a valid file
            bm._save_json()
    except Exception:  # noqa: BLE001
        traceback.print_exc()
    teardown()
if RUN_ERROR is not None:
    raise RUN_ERROR

## 8. Summary and ledger row

In [ ]:
runs = list(getattr(bm, "game_runs", []))
scores = {r.game_id: float(r.final_score or 0.0) for r in runs}
summary = {
    "experiment": CONFIG["experiment"],
    "submission": TRUE_SUBMISSION,
    "vllm_profile": SERVE["profile"],
    "vllm_attempts": SERVE["attempts"],
    "vllm_restarts": SERVE["restarts"],
    "throughput": SERVE["bench"],
    "games": len(runs),
    "mean_score": round(sum(scores.values()) / max(1, len(scores)), 4),
    "levels_completed": sum(r.levels_completed for r in runs),
    "levels_total": sum(r.number_of_levels for r in runs),
    "states": {s: sum(1 for r in runs if r.state == s) for s in {r.state for r in runs}},
    "per_game": scores,
    "solver": CONFIG["solver"],
    "wall_s": round(time.time() - NOTEBOOK_START_EPOCH),
}
(WORKING_DIR / "sarbloh_summary.json").write_text(json.dumps(summary, indent=1))
if not TRUE_SUBMISSION:
    for gid, s in sorted(scores.items()):
        print(f"{gid:24s} {s:7.2f}")
print("LEDGER_ROW " + json.dumps({k: summary[k] for k in (
    "experiment", "vllm_profile", "throughput", "games", "mean_score", "levels_completed", "levels_total", "wall_s")}))